# Linear Scan Register Allocation — 学习笔记

实现 Tiger 编译器中的线性扫描寄存器分配算法 (ref: `../tiger/ARCHITECTURE.md` 任务一)。

---

## 背景: 为什么需要寄存器分配?

### 问题的本质

编译器在生成代码时, 为了简化实现, 会假设有**无限多个**临时变量/虚拟寄存器可用:

```
r100 = a + b       # 编译器用 r100, r101, ... 随便取名
r101 = r100 * 2
r102 = r101 + c
print(r102)
```

但实际上 CPU 的物理寄存器数量是**有限的**:
- x86: ~8 个通用寄存器
- ARM64: ~30 个通用寄存器 (可用约 6-10 个)
- RISC-V: 32 个 (除去 zero/ra/sp/gp/tp, 可用 ~12 个)

**核心矛盾**: 编译器产出了 N 个虚拟寄存器 (N 可能几百、几千), 但只有 K 个物理寄存器 (K 很小)。
必须决定哪些虚拟寄存器共享同一个物理寄存器, 以及哪些被迫放到栈上 ("spill")。

### 类比: 宿舍与行李

想象你搬宿舍, 有 6 个柜子 (物理寄存器) 但 100 件行李 (虚拟寄存器)。
策略:
- 一段**时间内**不需要同时用的行李, 可以用同一个柜子 (寄存器复用)
- 柜子不够时, 剩下的只能放地上 (spill 到栈)
- 要用地上某个东西时, 先从柜子里腾出一个临时空间 (spill scratch reg),
  把地上东西搬进来, 用完再搬回去

### 两种主流算法

| 方法 | 思想 | 复杂度 | 代表 |
|------|------|--------|------|
| **图着色** | 构建 interference graph (冲突图), 节点=虚拟寄存器, 边=冲突; K-着色 | NP-complete (简化后 O(n log n)) | Chaitin, Briggs |
| **线性扫描** | 不构图, 直接按程序顺序扫描 live intervals, 贪婪分配 | O(n log n), 但常数小得多 | Poletto & Sarkar, 1999 |

线性扫描比图着色**快 ~2x**, 质量损失 ~5-10%, 是 JIT 编译器的主流选择 (V8, HotSpot, LLVM JIT)。

---

## 核心概念: Live Interval (活跃区间)

每个虚拟寄存器的 **Live Interval** 是从它被定义(写入)到它最后一次被使用(读取)之间的指令范围。

```
指令 0:  r100 = 3          ← r100 在这里被定义
指令 1:  r101 = r100 + 1   ← r100 在这里被使用 (之后不再用 r100)
指令 2:  push r101         ← r101 被使用 (之后 r101 死亡)
指令 3:  r102 = 10         ← r102 被定义
指令 4:  print(r102)       ← r102 被使用 (之后 r102 死亡)
```

```
Live Intervals:
  r100: [0, 1]     定义在0, 最后使用在1 — 区间长度=2
  r101: [1, 2]     定义在1, 最后使用在2 — 区间长度=2  
  r102: [3, 4]     定义在3, 最后使用在4 — 区间长度=2

可视化:
  指令:    0   1   2   3   4
  r100:  [####]               
  r101:       [####]           r101 与 r100 重叠! 不能共享同一个寄存器
  r102:               [####]   r102 不与任何人重叠, 可以复用 r100 或 r101 的寄存器
```

**关键直觉**: 如果两个 virtual register 的 live intervals **不重叠**, 它们就可以复用同一物理寄存器。
线性扫描就是按时间顺序逐个处理这些 interval, 贪婪地做出 register assignment 决策。

---

## 线性扫描算法 — 逐步图解

假设有 3 个物理寄存器 (R0, R1, R2), 需要分配以下 registers:

```
  r100: [0, 5]
  r101: [1, 6]
  r102: [2, 7]
  r103: [3, 4]     ← 短生命期, 夹在中间
```

### Step 1: 排序 (按 start 升序)

```
  [r100: 0-5] → [r101: 1-6] → [r102: 2-7] → [r103: 3-4]
```

### Step 2: 逐个处理

```
处理 r100[0,5]:  free=[R0,R1,R2]  → 分配给 R0
  active = [r100:R0] (按 end 排序: r100=5)

处理 r101[1,6]:  free=[R1,R2]     → 分配给 R1
  active = [r100:R0, r101:R1] (按 end: r100=5, r101=6)

处理 r102[2,7]:  free=[R2]         → 分配给 R2  
  active = [r100:R0, r101:R1, r102:R2] (按 end: r100=5, r101=6, r102=7)

处理 r103[3,4]:  free=[] 满了!
  → 先 expire: r100 的 end=5 > r103 的 start=3? 不, 5 > 3, 还活着
  → 检查 active 中最远的: r102 end=7
  → r102.end(7) > r103.end(4) → spill r102, 把 R2 给 r103!
     (r102 活得更久, 放栈上; r103 很短, 值得给寄存器)
  active = [r100:R0, r101:R1, r103:R2]
```

**关键决策**: 当寄存器不够用时, **spill 活得更远的那个**。
直觉: 如果我必须让一个人没柜子用, 我选那个需要柜子时间最长的 — 
让他费点事 (栈上), 柜子留给很快就能用完的人, 整体更高效。

这是"farthest-end heuristic" — 贪心地让生命周期最长的变量来做最多的 spill。
此启发式在实践中**接近最优**。

---

## 实现概览

```
Step 1: compute_live_intervals(instrs)
  → 扫描指令, 记录每个 vreg 首次定义 & 最后使用

Step 2: linear_scan(intervals, config)
  → 排序 → expire → alloc/spill → 输出分配结果

Step 3: rewrite_instructions(instrs, intervals, config)
  → 替换 vreg 名字 → 插入 spill load/store
```

下面我们逐一实现。

## 1. 数据结构定义

我们需要三个核心数据结构:

- **`LiveInterval`**: 表示一个虚拟寄存器的活跃区间, 以及分配决策 (分配哪个物理寄存器 / 是否 spill)
- **`Instr`**: 表示一条指令 (携带它定义和使用了哪些虚拟寄存器)
- **`RA_Config`**: 寄存器配置 (寄存器数量、名字、分配策略)

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Optional
import itertools

In [2]:
@dataclass
class LiveInterval:
    """虚拟寄存器的活跃区间

    start: 第一次被定义(写入)的指令索引
    end:   最后一次被使用(读取)的指令索引

    在区间 [start, end] 内, 寄存器的值必须保持。
    区间外的任何指令都不应该再读取该值。
    """
    vreg: str
    start: int
    end: int
    assigned_reg: Optional[int] = None  # 分配的物理寄存器编号
    spilled: bool = False                # 是否被溢出到栈
    spill_slot: Optional[int] = None     # 栈上的偏移量 (slot 编号)

    def __repr__(self):
        status = f" -> P{self.assigned_reg}" if not self.spilled else " [SPILLED]"
        return f"{self.vreg}[{self.start},{self.end}]{status}"


@dataclass
class Instr:
    """一条指令

    每条指令有 defs (定义/写入的虚拟寄存器集合) 和 uses (读取的虚拟寄存器集合)。
    例如 `r101 = r100 + 1` → defs={r101}, uses={r100}
    """
    idx: int                  # 指令在程序中的位置索引
    text: str                 # 汇编文本 (可读性)
    defs: set[str] = field(default_factory=set)
    uses: set[str] = field(default_factory=set)


def instr(idx, text, defs=(), uses=()):
    """快捷构造器"""
    return Instr(idx, text, set(defs), set(uses))

## 2. 计算 Live Intervals

**算法**: 单次扫描指令列表, 对每个虚拟寄存器维护两个值:
- `first_def[r]`: r 第一次作为 def 或 use 出现的指令索引
- `last_use[r]`: r 最后一次作为 def 或 use 出现的指令索引

最终 live interval = `[first_def[r], last_use[r]]`

**注意**: 这里用的是简化版的 liveness 分析 (不精确到基本块级别), 但足以说明线性扫描的原理。
生产级实现 (LLVM, V8) 会使用更精确的 liveness 分析 (dataflow analysis over basic blocks)。

In [3]:
def compute_live_intervals(instrs: list[Instr]) -> list[LiveInterval]:
    first_def: dict[str, int] = {}
    last_use: dict[str, int] = {}

    for i in instrs:
        for r in i.defs:
            if r not in first_def:
                first_def[r] = i.idx
            last_use[r] = max(last_use.get(r, -1), i.idx)
        for r in i.uses:
            if r not in first_def:
                first_def[r] = i.idx
            last_use[r] = max(last_use.get(r, -1), i.idx)

    intervals = []
    for r in first_def:
        s, e = first_def[r], last_use[r]
        if s <= e:
            intervals.append(LiveInterval(r, s, e))
    return intervals

## 3. 寄存器配置

Tiger 编译器支持两种后端, 对应不同的寄存器选择。

**VM 模式**: 使用 callee-saved 风格的 R0-R5 (模拟 VM 解释器), R6-R7 作为 spill scratch。

**ARM64 模式**: 使用 x19-x24 (callee-saved 寄存器, 跨 `bl` 调用安全), x25-x26 作为 spill scratch。

> **为什么 ARM64 用 callee-saved (x19-x24) 而不是 caller-saved (x9-x14)?**
>
> AAPCS64 调用约定规定: 函数调用 (`bl`) 后, caller-saved 寄存器 (x0-x18) 可能被破坏。
> Tiger 代码会调用 C 函数 (printInt, malloc...), 使用 callee-saved 寄存器确保变量在 `bl` 前后保持其值。
> 这是 Tiger 实现中踩过的坑 (详见 ARCHITECTURE.md Bug 4)。

In [4]:
@dataclass
class RA_Config:
    num_regs: int       # 物理寄存器总数 (含 spill scratch)
    num_alloc: int      # 可分配的数量 (索引 0..num_alloc-1)
    reg_names: list[str] # 寄存器名字列表


VM_CONFIG = RA_Config(
    num_regs=8, num_alloc=6,
    reg_names=["R0","R1","R2","R3","R4","R5","R6","R7"]
)

ARM64_CONFIG = RA_Config(
    num_regs=8, num_alloc=6,
    reg_names=["x19","x20","x21","x22","x23","x24","x25","x26"]
)

## 4. 线性扫描算法核心

这是整个算法的核心。让我们逐步理解每一步的设计动机。

### 算法伪代码

```python
intervals.sort(key=start)         # 按起始位置排序
active = []                       # 当前存活的 intervals, 按 end 升序
free = [0, 1, ..., num_alloc-1]   # 空闲物理寄存器栈

for cur in intervals:
    # Phase 1: 清理已死亡的 intervals
    for a in active:
        if a.end < cur.start:      # a 在 cur 开始前就已死亡
            free.append(a.reg)     # 释放寄存器
            active.remove(a)

    # Phase 2: 尝试分配
    if free:
        cur.reg = free.pop()       # 直接分配
        active.append(cur)
    else:
        farthest = max(active, key=end)  # active 中活得最远的
        if farthest.end > cur.end:
            # active 中某个活得更远 → spill 它
            spill(farthest)
            cur.reg = farthest.reg  # 接管其寄存器
            active.replace(farthest, cur)
        else:
            # cur 活得最远 → spill cur
            spill(cur)
```

### 为什么 active 按 end 排序?

- **Expire 时**: 我们只需要检查 end 最小的 — 如果它还没过期, 后面的更不可能过期 (不必全扫描)
- **Spill 时**: 我们需要找 end 最大的 — 遍历 active 取 max (复杂度 O(active_size))

如果 active 用**红黑树**维护 → Expire O(log n), Spill O(1) (按 max-end 索引)。
这里用 list + sort 简化实现, 演示原理。

In [5]:
def linear_scan(intervals: list[LiveInterval], config: RA_Config):
    """线性扫描寄存器分配

    输入:
        intervals: 所有虚拟寄存器的活跃区间
        config:    寄存器配置

    输出:
        (修改后的 intervals, 分配日志)
        每个 interval 的 assigned_reg / spilled / spill_slot 已被设置
    """
    sorted_intervals = sorted(intervals, key=lambda x: x.start)
    free_regs = list(range(config.num_alloc))
    active: list[LiveInterval] = []
    spill_slot_counter = itertools.count()
    logs: list[str] = []

    def expire_old_intervals(current: LiveInterval):
        nonlocal active, free_regs
        new_active = []
        for it in active:
            if it.end < current.start:
                free_regs.append(it.assigned_reg)
                logs.append(f"  free {it.vreg} (end={it.end} < {current.start}), release P{it.assigned_reg}")
            else:
                new_active.append(it)
        active = new_active

    for current in sorted_intervals:
        logs.append(f"\n[{current.vreg}] start={current.start}, end={current.end}")
        expire_old_intervals(current)

        if free_regs:
            # 有空闲寄存器 — 直接分配
            r = free_regs.pop()
            current.assigned_reg = r
            active.append(current)
            active.sort(key=lambda x: x.end)
            logs.append(f"  -> alloc P{r} ({config.reg_names[r]})")
        else:
            # 寄存器满了 — farthest-end heuristic
            farthest = max(active, key=lambda x: x.end)
            if farthest.end > current.end:
                logs.append(f"  spill {farthest.vreg} (end={farthest.end}) > {current.vreg} (end={current.end})")
                logs.append(f"  give P{farthest.assigned_reg} ({config.reg_names[farthest.assigned_reg]}) to {current.vreg}")
                current.assigned_reg = farthest.assigned_reg
                farthest.spilled = True
                farthest.slot = next(spill_slot_counter)
                farthest.assigned_reg = None
                active.remove(farthest)
                active.append(current)
                active.sort(key=lambda x: x.end)
            else:
                logs.append(f"  spill {current.vreg} (end={current.end}) >= farthest ({farthest.vreg} end={farthest.end})")
                current.spilled = True
                current.slot = next(spill_slot_counter)

    return sorted_intervals, logs

## 5. 指令改写 (Rewriting)

分配完成后, 我们需要**重写指令列表**:

1. 把虚拟寄存器名 (r100, r101...) 替换为物理寄存器名 (R0, R1..., 或 x19, x20...)
2. 对 spilled 的变量, 在使用前插入 **load** (从栈读到 spill scratch reg), 在定义后插入 **store** (写回栈)

**Spill scratch register** 是专门预留、不参与分配的物理寄存器 (如 R6/R7, x25/x26), 用作临时中转。

**一个已知局限**: 如果一条指令同时使用了多个 spilled 变量, 它们会共用同一个 spill scratch register,
后一个 load 会覆盖前一个 load 的值。生产实现需要用两个 spill scratch registers 交替使用, 或拆分指令。

In [6]:
def rewrite_instructions(instrs: list[Instr], intervals: list[LiveInterval],
                         config: RA_Config) -> list[str]:
    """重写指令列表, 输出最终的汇编代码"""
    vreg_map: dict[str, str] = {}
    spilled_set: set[str] = set()
    spill_slot: dict[str, int] = {}
    for it in intervals:
        if it.spilled:
            spilled_set.add(it.vreg)
            spill_slot[it.vreg] = it.slot
        else:
            vreg_map[it.vreg] = config.reg_names[it.assigned_reg]

    spill_scratch = config.reg_names[config.num_alloc]       # e.g. R6 / x25
    spill_scratch2 = config.reg_names[config.num_alloc + 1]  # e.g. R7 / x26

    output: list[str] = []
    for i in instrs:
        op = i.text

        # 对每个被使用的 spilled 变量: 插入 load
        for r in i.uses:
            if r in spilled_set:
                slot = spill_slot[r]
                output.append(f"    mov {spill_scratch}, [fp+{slot*8}]  ; reload {r}")
                op = op.replace(r, spill_scratch)

        # 替换非 spilled 的虚拟寄存器为物理寄存器名
        for r in i.defs:
            if r not in spilled_set:
                op = op.replace(r, vreg_map[r])
        for r in i.uses:
            if r not in spilled_set:
                op = op.replace(r, vreg_map[r])

        output.append(f"    {op}")

        # 对每个被定义的 spilled 变量: 插入 store
        for r in i.defs:
            if r in spilled_set:
                op2 = op.replace(r, spill_scratch)
                output[-1] = f"    {op2}"
                slot = spill_slot[r]
                output.append(f"    mov [fp+{slot*8}], {spill_scratch}  ; spill {r}")

    return output

---

## 测试验证

下面用 4 组测试数据验证算法:
1. **基本示例** — 来自 ARCHITECTURE.md 的文档示例, 验证基本逻辑
2. **无冲突** — 所有 intervals 不重叠, 应能复用寄存器
3. **强制 Spill** — 超过物理寄存器数量的同时存活变量, 验证 spill 机制
4. **复杂交错** — 部分重叠的 intervals, 模拟真实编译器输出

每组测试后都会打印 allocation log (分配过程) 和最终的改写指令。

## 测试用例 1: 基本示例

来自 ARCHITECTURE.md 第 103-114 行的例子:

```
r100: [0, 1]  定义在0, 最后使用在1
r101: [1, 2]  定义在1, 最后使用在2
r102: [3, 4]  定义在3, 最后使用在4
```

- r100 和 r101 **重叠** (都在指令1存活) → 不能共享同一寄存器
- r102 与前两者**不重叠** → 可以复用 r100 或 r101 的寄存器

In [7]:
instrs_1 = [
    instr(0, "r100 = 3",               defs=["r100"]),
    instr(1, "r101 = r100 + 1",        defs=["r101"], uses=["r100"]),
    instr(2, "push r101",              uses=["r101"]),
    instr(3, "r102 = 10",              defs=["r102"]),
    instr(4, "print(r102)",            uses=["r102"]),
]

intervals_1 = compute_live_intervals(instrs_1)
print("Live Intervals:")
for iv in sorted(intervals_1, key=lambda x: x.start):
    print(f"  {iv.vreg}: [{iv.start}, {iv.end}]")

print()
print("重叠关系: r100↔r101 重叠 | r102 独立")
print("预期: 需要至少 2 个物理寄存器 (r100/r101 不能共享, r102 复用)")

Live Intervals:
  r100: [0, 1]
  r101: [1, 2]
  r102: [3, 4]

重叠关系: r100↔r101 重叠 | r102 独立
预期: 需要至少 2 个物理寄存器 (r100/r101 不能共享, r102 复用)


In [8]:
result_1, log_1 = linear_scan(intervals_1, VM_CONFIG)
print("=== Allocation Log ===")
for line in log_1:
    print(line)

print("\n=== 分配结果 ===")
for iv in result_1:
    print(f"  {iv}")

print("\n=== 改写后的指令 ===")
for line in rewrite_instructions(instrs_1, result_1, VM_CONFIG):
    print(line)

=== Allocation Log ===

[r100] start=0, end=1
  -> alloc P5 (R5)

[r101] start=1, end=2
  -> alloc P4 (R4)

[r102] start=3, end=4
  free r100 (end=1 < 3), release P5
  free r101 (end=2 < 3), release P4
  -> alloc P4 (R4)

=== 分配结果 ===
  r100[0,1] -> P5
  r101[1,2] -> P4
  r102[3,4] -> P4

=== 改写后的指令 ===
    R5 = 3
    R4 = R5 + 1
    push R4
    R4 = 10
    print(R4)


## 测试用例 2: 无冲突场景 (链式依赖)

```
r100 → r101 → r102 → r103 → print(r103)
```

每一步都是用前一个值计算出新值, 所以每个 interval 只和下一个相邻的 interval 重叠。
理论上只需要 2 个寄存器就能搞定整个序列。

In [9]:
instrs_2 = [
    instr(0, "r100 = 1",        defs=["r100"]),
    instr(1, "r101 = r100",     defs=["r101"], uses=["r100"]),
    instr(2, "r102 = r101 + 1", defs=["r102"], uses=["r101"]),
    instr(3, "r103 = r102 + 1", defs=["r103"], uses=["r102"]),
    instr(4, "print(r103)",     uses=["r103"]),
]

intervals_2 = compute_live_intervals(instrs_2)
result_2, log_2 = linear_scan(intervals_2, VM_CONFIG)
print("=== Allocation Log ===")
for line in log_2:
    print(line)

print("\n=== 改写后的指令 ===")
for line in rewrite_instructions(instrs_2, result_2, VM_CONFIG):
    print(line)

print("\n预期: 交替使用 R0/R1, 只需 2 个物理寄存器")

=== Allocation Log ===

[r100] start=0, end=1
  -> alloc P5 (R5)

[r101] start=1, end=2
  -> alloc P4 (R4)

[r102] start=2, end=3
  free r100 (end=1 < 2), release P5
  -> alloc P5 (R5)

[r103] start=3, end=4
  free r101 (end=2 < 3), release P4
  -> alloc P4 (R4)

=== 改写后的指令 ===
    R5 = 1
    R4 = R5
    R5 = R4 + 1
    R4 = R5 + 1
    print(R4)

预期: 交替使用 R0/R1, 只需 2 个物理寄存器


## 测试用例 3: 强制 Spill

8 个变量在指令 0-7 依次定义, 但在指令 8 全部被使用。
→ 所有 8 个都在指令 8 存活 → 只有 6 个能拿到寄存器, 剩余 2 个必须 spill。

这验证算法的核心 spill 机制。

In [10]:
instrs_3 = [
    instr(0, "r100 = 1", defs=["r100"]),
    instr(1, "r101 = 2", defs=["r101"]),
    instr(2, "r102 = 3", defs=["r102"]),
    instr(3, "r103 = 4", defs=["r103"]),
    instr(4, "r104 = 5", defs=["r104"]),
    instr(5, "r105 = 6", defs=["r105"]),
    instr(6, "r106 = 7", defs=["r106"]),
    instr(7, "r107 = 8", defs=["r107"]),
    instr(8, "push r100+r101+r102+r103+r104+r105+r106+r107",
          uses=["r100","r101","r102","r103","r104","r105","r106","r107"]),
]

intervals_3 = compute_live_intervals(instrs_3)
result_3, log_3 = linear_scan(intervals_3, VM_CONFIG)
print("=== Allocation Log ===")
for line in log_3:
    print(line)

print("\n=== 分配结果 (注意哪些被 spill 了) ===")
for iv in result_3:
    print(f"  {iv}")

print("\n=== 改写后的指令 (spilled 变量会有 load/store) ===")
for line in rewrite_instructions(instrs_3, result_3, VM_CONFIG):
    print(line)

=== Allocation Log ===

[r100] start=0, end=8
  -> alloc P5 (R5)

[r101] start=1, end=8
  -> alloc P4 (R4)

[r102] start=2, end=8
  -> alloc P3 (R3)

[r103] start=3, end=8
  -> alloc P2 (R2)

[r104] start=4, end=8
  -> alloc P1 (R1)

[r105] start=5, end=8
  -> alloc P0 (R0)

[r106] start=6, end=8
  spill r106 (end=8) >= farthest (r100 end=8)

[r107] start=7, end=8
  spill r107 (end=8) >= farthest (r100 end=8)

=== 分配结果 (注意哪些被 spill 了) ===
  r100[0,8] -> P5
  r101[1,8] -> P4
  r102[2,8] -> P3
  r103[3,8] -> P2
  r104[4,8] -> P1
  r105[5,8] -> P0
  r106[6,8] [SPILLED]
  r107[7,8] [SPILLED]

=== 改写后的指令 (spilled 变量会有 load/store) ===
    R5 = 1
    R4 = 2
    R3 = 3
    R2 = 4
    R1 = 5
    R0 = 6
    R6 = 7
    mov [fp+0], R6  ; spill r106
    R6 = 8
    mov [fp+8], R6  ; spill r107
    mov R6, [fp+0]  ; reload r106
    mov R6, [fp+8]  ; reload r107
    push R5+R4+R3+R2+R1+R0+R6+R6


## 测试用例 4: 复杂交错存活

模拟真实编译器的输出 — 多个变量部分重叠, 生命周期长短不一。

```
  r100: [0, 2]       短命 — 加载后只用一次
  r101: [1, 3]       短命
  r102: [2, 8]       长命 — 早期计算, 分支后还要用
  r103: [3, 8]       长命
  r104: [4, 5]       短命
  r105: [5, 6]       最短命
  r106: [8, 9]       分支路径
  r107: [9, 10]      分支路径
```

有趣之处: r102/r103 在指令 8 再次被用, 但它们定义在指令 2/3,
中间经过了 4-7 的好几条指令。这意味着它们的生命期很长,
可能在某些分配策略下触发 spill。

In [11]:
instrs_4 = [
    instr(0,  "r100 = [fp+8]",      defs=["r100"]),
    instr(1,  "r101 = [fp+16]",     defs=["r101"]),
    instr(2,  "r102 = r100 * 2",    defs=["r102"], uses=["r100"]),
    instr(3,  "r103 = r101 + 1",    defs=["r103"], uses=["r101"]),
    instr(4,  "r104 = r102 + r103", defs=["r104"], uses=["r102","r103"]),
    instr(5,  "r105 = r104 < 8",    defs=["r105"], uses=["r104"]),
    instr(6,  "cmp r105, 0",        uses=["r105"]),
    instr(7,  "je done",            uses=[]),
    instr(8,  "r106 = r102 - r103", defs=["r106"], uses=["r102","r103"]),
    instr(9,  "r107 = r106 * 2",    defs=["r107"], uses=["r106"]),
    instr(10, "print(r107)",        uses=["r107"]),
]

intervals_4 = compute_live_intervals(instrs_4)

print("Live Intervals:")
for iv in sorted(intervals_4, key=lambda x: x.start):
    # 打印区间可视化
    bar = "·" * iv.start + "█" * (iv.end - iv.start + 1) + "·" * (10 - iv.end)
    print(f"  {iv.vreg:>5} [{iv.start:>2},{iv.end:>2}] |{bar}|")

result_4, log_4 = linear_scan(intervals_4, VM_CONFIG)
print("\n=== Allocation Log ===")
for line in log_4:
    print(line)

print("\n=== 改写后的指令 (VM) ===")
for line in rewrite_instructions(instrs_4, result_4, VM_CONFIG):
    print(line)

Live Intervals:
   r100 [ 0, 2] |███········|
   r101 [ 1, 3] |·███·······|
   r102 [ 2, 8] |··███████··|
   r103 [ 3, 8] |···██████··|
   r104 [ 4, 5] |····██·····|
   r105 [ 5, 6] |·····██····|
   r106 [ 8, 9] |········██·|
   r107 [ 9,10] |·········██|

=== Allocation Log ===

[r100] start=0, end=2
  -> alloc P5 (R5)

[r101] start=1, end=3
  -> alloc P4 (R4)

[r102] start=2, end=8
  -> alloc P3 (R3)

[r103] start=3, end=8
  free r100 (end=2 < 3), release P5
  -> alloc P5 (R5)

[r104] start=4, end=5
  free r101 (end=3 < 4), release P4
  -> alloc P4 (R4)

[r105] start=5, end=6
  -> alloc P2 (R2)

[r106] start=8, end=9
  free r104 (end=5 < 8), release P4
  free r105 (end=6 < 8), release P2
  -> alloc P2 (R2)

[r107] start=9, end=10
  free r102 (end=8 < 9), release P3
  free r103 (end=8 < 9), release P5
  -> alloc P5 (R5)

=== 改写后的指令 (VM) ===
    R5 = [fp+8]
    R4 = [fp+16]
    R3 = R5 * 2
    R5 = R4 + 1
    R4 = R3 + R5
    R2 = R4 < 8
    cmp R2, 0
    je done
    R2 = R3 - R5
    R

## 测试用例 5: ARM64 模式对比

同样数据用 ARM64 配置 (x19-x24 可分配, x25-x26 为 spill scratch) 验证。
分配结果应该与 VM 模式完全相同 (只是寄存器名字不同)。

In [12]:
result_4a, log_4a = linear_scan(intervals_4, ARM64_CONFIG)
print("=== 改写后的指令 (ARM64) ===")
for line in rewrite_instructions(instrs_4, result_4a, ARM64_CONFIG):
    print(line)

=== 改写后的指令 (ARM64) ===
    x24 = [fp+8]
    x23 = [fp+16]
    x22 = x24 * 2
    x24 = x23 + 1
    x23 = x22 + x24
    x21 = x23 < 8
    cmp x21, 0
    je done
    x21 = x22 - x24
    x24 = x21 * 2
    print(x24)


## 验证: 分配正确性检查

验证两个核心不变量:
1. **无冲突**: 任意两个重叠的 intervals 不能分配到同一物理寄存器
2. **寄存器不溢出**: 分配编号不超过 `num_alloc`

In [13]:
def verify_allocation(intervals: list[LiveInterval], config: RA_Config) -> bool:
    """验证分配结果无冲突"""
    allocated = [iv for iv in intervals if not iv.spilled]
    spilled = [iv for iv in intervals if iv.spilled]

    # 检查 1: 重叠区间不能共享物理寄存器
    for i, a in enumerate(allocated):
        for b in allocated[i+1:]:
            if a.assigned_reg == b.assigned_reg:
                if a.start <= b.end and b.start <= a.end:
                    print(f"  ERROR: {a.vreg} and {b.vreg} overlap but share P{a.assigned_reg}")
                    return False

    # 检查 2: 使用的物理寄存器编号不超过 num_alloc
    used_regs = set(iv.assigned_reg for iv in allocated)
    if any(r >= config.num_alloc for r in used_regs):
        print(f"  ERROR: register overflow {used_regs} (max alloc={config.num_alloc-1})")
        return False

    print(f"  OK: {len(allocated)} allocated, {len(spilled)} spilled, "
          f"{len(used_regs)} physical regs used (max {config.num_alloc})")
    return True


for name, ivs in [("Test 1", result_1), ("Test 2", result_2),
                   ("Test 3", result_3), ("Test 4", result_4)]:
    print(f"{name}:", end=" ")
    verify_allocation(ivs, VM_CONFIG)

Test 1:   OK: 3 allocated, 0 spilled, 2 physical regs used (max 6)
Test 2:   OK: 4 allocated, 0 spilled, 2 physical regs used (max 6)
Test 3:   OK: 6 allocated, 2 spilled, 6 physical regs used (max 6)
Test 4:   OK: 8 allocated, 0 spilled, 4 physical regs used (max 6)


---

## 总结

| 步骤 | 做什么 | 为什么 |
|------|--------|--------|
| **Live Interval 计算** | 扫描指令, 记录每个 vreg 的首次 def 和最终 use | 确定每个变量的"活跃窗口" |
| **按 start 排序** | 按 intervals 起点升序排列 | 保证按程序顺序处理 (线性扫描) |
| **Expire** | 释放 end < current.start 的 active intervals | 已死亡的变量不再占用寄存器 |
| **分配** | 有空闲寄存器 → 直接分配 | 贪婪策略: 能用就用 |
| **Spill** | 寄存器满时, spill end 最远的 | farthest-end heuristic: 让活最久的变量去栈上, 短命变量留在寄存器 |
| **Rewrite** | 替换 vreg 名字, 插入 load/store | 实现 spill: 用 scratch reg 中转, 从栈上 load 进来 / store 回去 |

### 线性扫描 vs 图着色

| | 线性扫描 | 图着色 |
|------|------|------|
| 核心数据结构 | Interval (区间) | Interference Graph (冲突图) |
| 复杂度 | O(n log n), 常数小 | O(n²) 构图 + O(n log n) 着色, 常数大 |
| 质量 | 略差 (~5-10% 更多 spill) | 理论上更优 |
| 适用场景 | JIT 编译器 (V8, HotSpot), 移动端 | 离线编译器 (GCC, LLVM -O2) |
| 代表论文 | Poletto & Sarkar, PLDI 1999 | Chaitin, 1981; Briggs, 1994 |

### Tiger 中的实际应用

文件: `regalloc.c` / `regalloc.h`, `codegen_arm64.c`

- VM 模式下 R0-R5 为分配寄存器, R6-R7 为 spill scratch
- ARM64 下 x19-x24 为分配寄存器 (callee-saved, 跨 bl 安全), x25-x26 为 spill scratch
- 当前 8-queens 程序在 6 个寄存器下产生 0 spill (变量生命周期短, 复合度低)
- 已知问题: 多条 spilled 操作数在同一指令中时, 共享 spill 寄存器导致值覆盖